In [2]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot
from tqdm import tqdm

from model.dataset import BagsDataset, custom_collate_fn

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
adata = sc.read_h5ad('../example.h5ad')

In [5]:
dataset = BagsDataset(adata, radius= 200, immune_cell='tcell')

Creating Bags with radius 200: 100%|█████████████████████████| 2640/2640 [00:00<00:00, 10577.01it/s]

Total bags created: 1359
Average instances per bag: 6


In [6]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)


In [7]:
distances, gene_expressions, labels, core_index = next(iter(dataloader))

In [8]:
class Sparsemax(nn.Module):
    def __init__(self, dim=None):
        super(Sparsemax, self).__init__()
        self.dim = -1 if dim is None else dim

    def forward(self, input):
        input = input.transpose(0, self.dim)
        original_size = input.size()
        input = input.reshape(input.size(0), -1)
        input = input.transpose(0, 1)
        dim = 1

        number_of_logits = input.size(dim)

        input = input - torch.max(input, dim=dim, keepdim=True)[0].expand_as(input)

        zs = torch.sort(input=input, dim=dim, descending=True)[0]
        range = torch.arange(start=1, end=number_of_logits + 1, step=1, device=input.device, dtype=input.dtype).view(1, -1)
        range = range.expand_as(zs)

        bound = 1 + range * zs
        cumulative_sum_zs = torch.cumsum(zs, dim)
        is_gt = torch.gt(bound, cumulative_sum_zs).type(input.type())
        k = torch.max(is_gt * range, dim, keepdim=True)[0]

        zs_sparse = is_gt * zs

        taus = (torch.sum(zs_sparse, dim, keepdim=True) - 1) / k
        taus = taus.expand_as(input)

        output = F.relu(input - taus)

        output = output.transpose(0, 1)
        output = output.reshape(original_size)
        output = output.transpose(0, self.dim)

        return output

In [9]:
model = Sparsemax()
output = model(gene_expressions[0])
output

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [10]:
print(torch.count_nonzero(gene_expressions[0] == 0)) 
print(torch.count_nonzero(output == 0)) 

tensor(69823)
tensor(77512)


In [11]:
class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(1.0))
        self.sparsemax = Sparsemax(dim=0)
    
    def forward(self, x):
        #print(x)
        a = torch.sigmoid(self.a)  
        x = self.sparsemax(-a * x)
        return x

In [12]:
model = Distance()
output = model(distances[0])
print(output)

tensor([[0.0000],
        [0.0000],
        [0.5000],
        [0.5000]], grad_fn=<TransposeBackward0>)


In [13]:
class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0))
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        b = torch.sigmoid(self.b)
        x = b * x
        x = self.softmax(x)
        return x


In [14]:
gene_expressions[0].shape

torch.Size([4, 19379])

In [15]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output.shape)

torch.Size([4, 19379])


In [16]:
class Immunogenicity(nn.Module):
    def __init__(self):
        super(Immunogenicity, self).__init__()
        self.ig = nn.Parameter(torch.randn(19379))
    
    def forward(self):
        
        return self.ig.T

In [17]:
model = Immunogenicity()

output = model()
output

/tmp/ipykernel_5246/828137327.py:8: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3675.)
  return self.ig.T


tensor([ 0.3282, -0.7017, -1.8412,  ...,  0.3982, -0.3480, -0.9482],
       grad_fn=<PermuteBackward0>)

In [18]:
print(output.shape)

torch.Size([19379])


In [19]:
class MIL(nn.Module):
    def __init__(self):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.Immunogenicity = Immunogenicity()
        #self.pooling = MILPooling()
    
    def forward(self, distances, gene_expressions):
        for distance, gene_expression in zip(distances, gene_expressions):
            #print(distance.size())
            distance = self.distance(distance)
            #print("xxxxxxx")
            gene_expression = self.gene_expression(gene_expression)
            #print(gene_expression.shape)
            immunogenicity = self.Immunogenicity()
            #print(affinity,shape)
            #for j in range(len(gene_expression)):
                #zj = gene_expression[j, :] 
                #z.append(zj)
            z = gene_expression @ immunogenicity
            #print(z)
            #print("***")
            
            #print(z.shape)
            #print("***")
            #print(distance.squeeze().shape)
            #print(z)
            #print(z)
            z = z.unsqueeze(1)
            #print(z)
            bag_output = distance * z
            bag_output = torch.sum(bag_output, dim=0)
            #print(bag_output)
            bag_output = torch.sigmoid(bag_output)
        return bag_output
    




In [20]:
model = MIL()
output = model(distances, gene_expressions)

In [21]:
output

tensor([0.5932], grad_fn=<SigmoidBackward0>)

In [22]:
labels[0]


tensor(0.)

In [23]:
make_dot(output, params=dict(model.named_parameters())).render("MIL_computational_graph", format="png")


'MIL_computational_graph.png'

In [24]:
def model_summary(model, input_data):
    def register_hook(module):
        def hook(module, input, output):
            class_name = str(module.__class__).split(".")[-1].split("'")[0]
            module_idx = len(summary)

            m_key = "%s-%i" % (class_name, module_idx + 1)
            summary[m_key] = OrderedDict()
            
            if input:
                summary[m_key]["input_shape"] = list(input[0].size())
            else:
                summary[m_key]["input_shape"] = "No Input"
                
            try:
                summary[m_key]["output_shape"] = list(output.size())
            except AttributeError:
                summary[m_key]["output_shape"] = "No Output Size"
                
            summary[m_key]["trainable"] = any(p.requires_grad for p in module.parameters())
            summary[m_key]["nb_params"] = sum(p.numel() for p in module.parameters() if p.requires_grad)

        if not isinstance(module, nn.Sequential) and \
           not isinstance(module, nn.ModuleList) and \
           not (module == model):
            hooks.append(module.register_forward_hook(hook))

    # Create properties
    summary = OrderedDict()
    hooks = []

    # Register hook
    model.apply(register_hook)

    # Make a forward pass
    model(*input_data)

    # Remove these hooks
    for h in hooks:
        h.remove()

    # Print out the summary
    print("----------------------------------------------------------------")
    print("{:>20} {:>25} {:>15}".format("Layer (type)", "Output Shape", "Param #"))
    print("================================================================")
    total_params = 0
    total_output = 0
    trainable_params = 0
    for layer in summary:
        # Input shape, output shape, and number of parameters
        line_new = "{:>20} {:>25} {:>15}".format(
            layer,
            str(summary[layer]["output_shape"]),
            "{0:,}".format(summary[layer]["nb_params"]),
        )
        total_params += summary[layer]["nb_params"]
        if summary[layer]["output_shape"] != "Error" and summary[layer]["output_shape"] != "No Output Size":
            total_output += np.prod(summary[layer]["output_shape"])
        if "trainable" in summary[layer] and summary[layer]["trainable"]:
            trainable_params += summary[layer]["nb_params"]
        print(line_new)

    # Total params
    total_input_size = np.sum([np.prod(i.shape) for i in input_data if isinstance(i, torch.Tensor)]) * 4. / (1024 ** 2.)
    total_output_size = 2. * total_output * 4. / (1024 ** 2.)  # x2 for gradients
    total_params_size = total_params * 4. / (1024 ** 2.)
    total_size = total_params_size + total_output_size + total_input_size

    print("================================================================")
    print("Total params: {0:,}".format(total_params))
    print("Trainable params: {0:,}".format(trainable_params))
    print("Non-trainable params: {0:,}".format(total_params - trainable_params))
    print("----------------------------------------------------------------")
    print("Input size (MB): %0.2f" % total_input_size)
    print("Forward/backward pass size (MB): %0.2f" % total_output_size)
    print("Params size (MB): %0.2f" % total_params_size)
    print("Estimated Total Size (MB): %0.2f" % total_size)
    print("----------------------------------------------------------------")

In [25]:
model_summary(model, (distances, gene_expressions))


----------------------------------------------------------------
        Layer (type)              Output Shape         Param #
         Sparsemax-1                    [4, 1]               0
          Distance-2                    [4, 1]               1
           Softmax-3                [4, 19379]               0
   Gene_expression-4                [4, 19379]               1
    Immunogenicity-5                   [19379]          19,379
Total params: 19,381
Trainable params: 19,381
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 1.33
Params size (MB): 0.07
Estimated Total Size (MB): 1.40
----------------------------------------------------------------


In [26]:
model = MIL()
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.to(device)

MIL(
  (distance): Distance(
    (sparsemax): Sparsemax()
  )
  (gene_expression): Gene_expression(
    (softmax): Softmax(dim=-1)
  )
  (Immunogenicity): Immunogenicity()
)

In [28]:
num_epochs = 1

for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()
            output = model(distances, gene_expressions)
            #print(output)
            #print(label)
            loss = criterion(output, label)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')


Epoch 1/1: 100%|██████████| 1359/1359 [00:03<00:00, 353.58batch/s, loss=0.311]

Epoch [1/1], Loss: 0.7843


In [29]:
def predict(model, adata, device, radius=200, max_instances=None):
    model.eval()
    
    dataset = BagsDataset(adata_radius_input=adata, immune_cell='tcell', radius=radius, max_instances=max_instances)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=custom_collate_fn)
    
    predictions = np.full(len(adata.obs), np.nan)  # Initialize predictions array with NaNs

    with torch.no_grad():
        with tqdm(dataloader, unit="batch") as tepoch:
            for distances, gene_expressions, _, core_idx in tepoch:
                tepoch.set_description("Predicting")
                
                # Move data to the device
                distances = [d.to(device) for d in distances]
                gene_expressions = [g.to(device) for g in gene_expressions]
                
                output = model(distances, gene_expressions)
                predictions[core_idx.item()] = output.cpu().numpy().flatten()

    adata.obs['tcr_predict'] = predictions
    return adata


In [30]:
adata = predict(model, adata,radius=20, device=device)

Creating Bags with radius 20: 100%|██████████████████████████| 2640/2640 [00:00<00:00, 14262.17it/s]


Total bags created: 1359
Average instances per bag: 2


Predicting:   0%|          | 0/1359 [00:00<?, ?batch/s]/tmp/ipykernel_5246/3842026426.py:19: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  predictions[core_idx.item()] = output.cpu().numpy().flatten()
Predicting: 100%|██████████| 1359/1359 [00:01<00:00, 1012.60batch/s]


In [31]:
adata.obs.

,in_tissue,array_row,array_col,n_counts,X,Y,tcr,cell_type,tcr_predict
AAACAAGTATCTCCCA-1,1,50,102,5625.0,5000,10200,1,1,0.477785
AAACACCAATAACTGC-1,1,59,19,9159.0,5900,1900,1,0,NaN
AAACAGAGCGACTCCT-1,1,14,94,2660.0,1400,9400,0,1,0.521690
AAACAGGGTCTATATT-1,1,47,13,9020.0,4700,1300,1,1,0.218820
AAACAGTGTTCCTGGG-1,1,73,43,9714.0,7300,4300,0,0,NaN
...,...,...,...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,7260.0,3100,7700,0,1,0.477785
TTGTTTCACATCCAGG-1,1,58,42,7510.0,5800,4200,1,1,0.218819
TTGTTTCATTAGTCTA-1,1,60,30,5742.0,6000,3000,1,1,0.386546
TTGTTTCCATACAACT-1,1,45,27,20118.0,4500,2700,0,1,0.218819
